In [ ]:
"""
==============================
ENGLISH TRANSLATION + EXPLANATION
==============================

This script calculates performance scores for different roles (EVs, ECs, SDRs)
based on business KPIs. It includes data preparation, normalization, benchmarking,
and multiple scoring methodologies.

------------------------------
IMPORTS
------------------------------
- pandas: data manipulation
- numpy: numerical operations
- datetime: timestamp generation
"""

import pandas as pd  # FINAL OFFICIAL VERSION - OPTIMIZED - WITHOUT EARLY CHURN + FULL STRUCTURE + RETENTION
import numpy as np
from datetime import datetime

# =====================================================
# 1. GENERAL CONFIGURATIONS AND BUSINESS RULES
# =====================================================

# Data sources
ARQUIVO_EXCEL = r"C:\Users\FabianaKuhlmann\Downloads\Programa High Performers (Phantom Share) (2).xlsx"
ARQUIVO_HISTORICO_MEDIAS = "Historico_Medias_Cluster.csv"

# Sheet names
ABA_EVS, ABA_ECS, ABA_SDRS, ABA_METAS, ABA_P10 = "EVs", "ECs", "SDRs", "%Ating_Metas", "Página10"

# Analysis period
DATA_INICIO, DATA_FIM = "2026-01-01", "2026-03-31"

# Column standardization
COL_PERIODO, COL_FRANQUIA, COL_CLUSTER = "Adjusted Period", "Adjusted Franchise", "Adjusted Cluster"
COL_INDICADOR, COL_RESULTADO, COL_COLABORADOR, COL_STATUS = "Adjusted Indicator", "Result", "Employee Name", "CURRENT STATUS"
COL_META_COMERCIAL, COL_NMRR_UNIDADE = "Commercial Target", "NMRR"

# Business rules and thresholds
SHARE_FIXO_PADRAO, TETO_SOLO, SCORE_MINIMO = 0.40, 0.60, 70
SHARE_FIXO_SDR, TETO_SOLO_SDR = 0.50, 0.70
PCT_META_MINIMO, META_VS_MEDIA_MINIMA = 0.6, 100

# KPI weights (Early Churn removed and redistributed)
PESOS_EVS = {"NMRR": 0.5, "Demos": 0.25, "Conversion": 0.25}
PESOS_ECS = {"Accountant NMRR": 0.4, "Accountant Leads": 0.2, "Referring Accountants": 0.2, "Meetings": 0.2}
PESOS_SDRS = {"Scheduled Leads": 0.50, "Worked Leads": 0.25, "Scheduling Rate (%)": 0.25}

# No inverse indicators after removing Early Churn
INDICADORES_INVERSOS = []

# =====================================================
# 2. FUNCTION: CALCULATE CLUSTER AVERAGES
# =====================================================

def get_medias_cluster(df, indicadores, perfil_nome):
    """
    Calculates average KPI performance per cluster.

    Steps:
    1. Filters valid indicators present in dataset
    2. Calculates mean result per cluster and indicator
    3. Pivots data to wide format
    4. Adds profile name and calculation timestamp
    """

    indicadores_presentes = [i for i in indicadores if i in df[COL_INDICADOR].unique()]
    if not indicadores_presentes:
        return pd.DataFrame()

    medias = (
        df[df[COL_INDICADOR].isin(indicadores_presentes)]
        .groupby([COL_CLUSTER, COL_INDICADOR])[COL_RESULTADO]
        .mean()
        .reset_index()
    )

    pivot_medias = medias.pivot(
        index=COL_CLUSTER,
        columns=COL_INDICADOR,
        values=COL_RESULTADO
    ).reset_index()

    pivot_medias.insert(0, "Profile", perfil_nome)
    pivot_medias["Calculation_Date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    return pivot_medias

# =====================================================
# 3. FUNCTION: CALCULATE ALL PERFORMANCE SCORES
# =====================================================

def calcular_todos_racionais(df, pesos_dict, df_bench):
    """
    Core scoring engine.

    This function:
    - Restructures raw data
    - Aggregates performance per employee
    - Calculates normalized and percentile scores
    - Benchmarks against cluster averages
    - Produces multiple scoring methodologies
    """

    indicadores_config = list(pesos_dict.keys())

    # Pivot raw data into structured format
    base = df.pivot_table(
        index=[COL_COLABORADOR, COL_FRANQUIA, COL_CLUSTER, COL_PERIODO, COL_STATUS],
        columns=COL_INDICADOR,
        values=COL_RESULTADO,
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    periodos_colab = base[[COL_COLABORADOR, COL_PERIODO]].copy()
    indicadores_reais = [i for i in indicadores_config if i in base.columns]

    # Aggregate per employee (mean performance + latest status)
    res = base.groupby(
        [COL_COLABORADOR, COL_FRANQUIA, COL_CLUSTER],
        as_index=False
    ).agg({
        **{ind: "mean" for ind in indicadores_reais},
        COL_STATUS: "last"
    })

    # Create percentile and normalized scores
    for ind in indicadores_reais:
        rank_asc = False if ind in INDICADORES_INVERSOS else True

        base[f"pct_{ind}"] = (
            base.groupby([COL_CLUSTER, COL_PERIODO])[ind]
            .rank(pct=True, ascending=rank_asc) * 100
        )

        base[f"norm_{ind}"] = (
            base.groupby([COL_CLUSTER, COL_PERIODO])[ind]
            .transform(lambda x: (x / (x.max() if x.max() != 0 else 1)) * 100)
        )

    # Aggregate score components
    res_scores_aux = base.groupby([COL_COLABORADOR]).agg(
        **{f"pct_{ind}": (f"pct_{ind}", "mean") for ind in indicadores_reais},
        **{f"norm_{ind}": (f"norm_{ind}", "mean") for ind in indicadores_reais}
    ).reset_index()

    # Merge with cluster benchmark
    res = res.merge(df_bench, on=COL_CLUSTER, suffixes=('_colab', '_cluster'))

    # Compare employee vs cluster average
    for ind in indicadores_reais:
        if f"{ind}_colab" in res.columns and f"{ind}_cluster" in res.columns:
            res[f"{ind}_vs_Media_Cluster"] = (
                res[f"{ind}_colab"] / (res[f"{ind}_cluster"].replace(0, 1))
            ) * 100

    res = res.merge(res_scores_aux, on=COL_COLABORADOR)

    # Ensure all expected columns exist
    for ind in indicadores_config:
        if f"pct_{ind}" not in res.columns:
            res[f"pct_{ind}"] = 0
        if f"norm_{ind}" not in res.columns:
            res[f"norm_{ind}"] = 0
        if f"{ind}_vs_Media_Cluster" not in res.columns:
            res[f"{ind}_vs_Media_Cluster"] = 0
        if f"{ind}_colab" not in res.columns:
            res[f"{ind}_colab"] = 0

    # Final scoring models
    res["score_R1_weighted_percentile"] = sum(
        res[f"pct_{ind}"] * pesos_dict[ind] for ind in indicadores_config
    )

    res["score_R2_simple_percentile"] = res[
        [f"pct_{ind}" for ind in indicadores_config]
    ].mean(axis=1)

    res["score_R3_weighted_normalized"] = sum(
        res[f"norm_{ind}"] * pesos_dict[ind] for ind in indicadores_config
    )

    res["score_R4_simple_normalized"] = res[
        [f"norm_{ind}" for ind in indicadores_config]
    ].mean(axis=1)

    res["score_R5_weighted_vs_cluster"] = sum(
        res[f"{ind}_vs_Media_Cluster"] * pesos_dict[ind] for ind in indicadores_config
    )

    res["score_R5_simple_vs_cluster"] = res[
        [f"{ind}_vs_Media_Cluster" for ind in indicadores_config]
    ].mean(axis=1)

    # Extract NMRR metric
    nmrr_key = next(
        (k for k in ["NMRR", "Accountant NMRR"] if k in base.columns),
        None
    )

    res["avg_NMRR_per_employee"] = (
        res[f"{nmrr_key}_colab"]
        if (nmrr_key and f"{nmrr_key}_colab" in res.columns)
        else 0
    )

    return res, periodos_colab

# =====================================================
# 4. FUNCTION: DEFINE PERFORMANCE REASON
# =====================================================

def definir_motivo_generico(row, col_score, meta_score, col_piso, col_share, tipo_share_nome, unidade_bateu_meta):
    """
    Generates a human-readable explanation for performance outcome.

    Returns:
    - "All criteria met" if everything is satisfied
    - Otherwise, a combination of failure reasons
    """

    if unidade_bateu_meta == "Sim" and row[col_share] >= (row[col_piso] - 0.001) and row[col_score] >= meta_score:
        return "All criteria met"

    motivos = []

    if unidade_bateu_meta == "Não":
        motivos.append("Unit < 60%")

    if row[col_share] < (row[col_piso] - 0.001):
        motivos.append(f"{tipo_share_nome} < Threshold")

    if row[col_score] < meta_score:
        motivos.append(f"Score < {meta_score}")

    return " / ".join(motivos)

"""
====================================================
FINAL CONSOLIDATION + MONTHLY AGGREGATION (ENGLISH)
====================================================

This section is responsible for:
- Building the final consolidated dataset per employee
- Applying eligibility rules (Top Talent logic)
- Handling SDR-specific logic (based on Leads instead of NMRR)
- Generating monthly aggregated views of performance

Key concepts:
- Share: individual contribution vs unit total
- Thresholds ("trava"): minimum required participation
- Eligibility: combination of performance + business constraints
"""

# =====================================================
# 1. FINAL CONSOLIDATION (GENERAL ROLES: EVs, ECs)
# =====================================================

def preparar_final_consolidado(df_score, df_origem, perfil, ind_nmrr, meta_periodo, hc_ref):
    """
    Builds the final dataset per employee, including:
    - NMRR share calculation
    - Eligibility rules (R1–R5)
    - Top Talent classification
    """

    # Total NMRR per employee
    nmrr_colab_total = (
        df_origem[df_origem[COL_INDICADOR] == ind_nmrr]
        .groupby(COL_COLABORADOR)[COL_RESULTADO]
        .sum()
        .reset_index(name="nmrr_total_colab")
    )

    # Merge all required datasets
    final = (
        df_score.merge(nmrr_colab_total, on=COL_COLABORADOR, how='left')
                .merge(meta_periodo, on=COL_FRANQUIA)
                .merge(hc_ref, on=COL_FRANQUIA, how='left')
    )

    # Calculate share (individual vs unit)
    final["share_nmrr_real"] = final["nmrr_total_colab"].fillna(0) / (
        final["nmrr_unidade_total"].replace(0, 1)
    )

    # Fixed and dynamic thresholds
    final["trava_fixa"] = np.where(
        final['hc_total_unidade'] <= 1,
        TETO_SOLO,
        SHARE_FIXO_PADRAO
    )

    final["trava_dinamica_hc"] = (
        1 / final['hc_total_unidade'].fillna(1)
    ).clip(upper=TETO_SOLO)

    final["perfil"] = perfil

    # -----------------------------
    # Eligibility R1 to R4 (Fixed threshold)
    # -----------------------------
    mapping_scores = {
        "R1": "score_R1_percentil_ponderado",
        "R2": "score_R2_percentil_simples",
        "R3": "score_R3_direto_ponderado",
        "R4": "score_R4_direto_simples"
    }

    for r, col_s in mapping_scores.items():
        final[f"Elegivel_{r}"] = np.where(
            (final["unidade_bateu_meta"] == "Sim") &
            (final["share_nmrr_real"] >= (final["trava_fixa"] - 0.001)) &
            (final[col_s] >= SCORE_MINIMO),
            "Eligible",
            "Not Eligible"
        )

        final[f"Eligibility_Reason_{r}"] = final.apply(
            lambda x: definir_motivo_generico(
                x,
                col_s,
                SCORE_MINIMO,
                "trava_fixa",
                "share_nmrr_real",
                "NMRR",
                x["unidade_bateu_meta"]
            ),
            axis=1
        )

    # -----------------------------
    # R5 (Top Talent logic - dynamic HC threshold)
    # -----------------------------
    final["TopTalent_Official"] = (
        (final["unidade_bateu_meta"] == "Sim") &
        (final["share_nmrr_real"] >= (final["trava_dinamica_hc"] - 0.001)) &
        (final["score_R5_vs_media_cluster_ponderado"] >= META_VS_MEDIA_MINIMA)
    )

    final["TopTalent_Official_simple"] = (
        (final["unidade_bateu_meta"] == "Sim") &
        (final["share_nmrr_real"] >= (final["trava_dinamica_hc"] - 0.001)) &
        (final["score_R5_vs_media_cluster_simples"] >= META_VS_MEDIA_MINIMA)
    )

    # Reasons for R5
    final["TopTalent_Reason_R5"] = final.apply(
        lambda x: definir_motivo_generico(
            x,
            "score_R5_vs_media_cluster_ponderado",
            META_VS_MEDIA_MINIMA,
            "trava_dinamica_hc",
            "share_nmrr_real",
            "NMRR",
            x["unidade_bateu_meta"]
        ),
        axis=1
    )

    final["TopTalent_Reason_R5_simple"] = final.apply(
        lambda x: definir_motivo_generico(
            x,
            "score_R5_vs_media_cluster_simples",
            META_VS_MEDIA_MINIMA,
            "trava_dinamica_hc",
            "share_nmrr_real",
            "NMRR",
            x["unidade_bateu_meta"]
        ),
        axis=1
    )

    final["business_rule_share"] = final["trava_dinamica_hc"]

    return final


# =====================================================
# 2. FINAL CONSOLIDATION FOR SDRs (R7 LOGIC)
# =====================================================

def preparar_final_sdrs_r7(df_score, df_origem, df_p10_raw, perfil, meta_periodo):
    """
    SDR-specific consolidation:
    - Uses Leads instead of NMRR
    - Applies same eligibility logic adapted to SDR KPIs
    """

    # Headcount per unit
    dist_sdr = df_origem.groupby(COL_FRANQUIA)[COL_COLABORADOR] \
        .nunique().reset_index(name='hc_total_unidade')

    # Clean franchise names
    df_p10_c = df_p10_raw.copy()
    df_p10_c[COL_FRANQUIA] = (
        df_p10_c[COL_FRANQUIA]
        .astype(str)
        .str.replace(r"^omie\s+", "", regex=True, case=False)
        .str.strip()
    )

    # Total leads per unit
    unidade_leads = df_p10_c.groupby(COL_FRANQUIA)["Leads Agendados"] \
        .sum().reset_index(name="total_leads_unidade_p10")

    # Individual leads
    leads_indiv = (
        df_origem[df_origem[COL_INDICADOR] == "Leads Agendados"]
        .groupby(COL_COLABORADOR)[COL_RESULTADO]
        .sum()
        .reset_index(name="nmrr_total_colab")
    )

    # Merge
    final = (
        df_score.merge(meta_periodo, on=COL_FRANQUIA)
                .merge(dist_sdr, on=COL_FRANQUIA)
                .merge(leads_indiv, on=COL_COLABORADOR, how="left")
                .merge(unidade_leads, on=COL_FRANQUIA, how="left")
    )

    # Fill missing values
    final["nmrr_total_colab"] = final["nmrr_total_colab"].fillna(0)
    final["total_leads_unidade_p10"] = final["total_leads_unidade_p10"].fillna(0)

    # Share calculation (based on leads)
    final["share_nmrr_real"] = np.where(
        final["total_leads_unidade_p10"] > 0,
        final["nmrr_total_colab"] / final["total_leads_unidade_p10"],
        0
    )

    # Thresholds
    final["trava_fixa"] = np.where(
        final['hc_total_unidade'] <= 1,
        TETO_SOLO_SDR,
        SHARE_FIXO_SDR
    )

    final["trava_dinamica_hc"] = (
        1 / final['hc_total_unidade'].fillna(1)
    ).clip(upper=TETO_SOLO_SDR)

    final["perfil"] = perfil

    # R1–R4 eligibility
    mapping_scores = {
        "R1": "score_R1_percentil_ponderado",
        "R2": "score_R2_percentil_simples",
        "R3": "score_R3_direto_ponderado",
        "R4": "score_R4_direto_simples"
    }

    for r, col_s in mapping_scores.items():
        final[f"Elegivel_{r}"] = np.where(
            (final["unidade_bateu_meta"] == "Sim") &
            (final["share_nmrr_real"] >= (final["trava_fixa"] - 0.001)) &
            (final[col_s] >= SCORE_MINIMO),
            "Eligible",
            "Not Eligible"
        )

        final[f"Eligibility_Reason_{r}"] = final.apply(
            lambda x: definir_motivo_generico(
                x,
                col_s,
                SCORE_MINIMO,
                "trava_fixa",
                "share_nmrr_real",
                "Leads",
                x["unidade_bateu_meta"]
            ),
            axis=1
        )

    # R5 Top Talent
    final["TopTalent_Official"] = (
        (final["unidade_bateu_meta"] == "Sim") &
        (final["share_nmrr_real"] >= (final["trava_dinamica_hc"] - 0.001)) &
        (final["score_R5_vs_media_cluster_ponderado"] >= META_VS_MEDIA_MINIMA)
    )

    final["TopTalent_Official_simple"] = (
        (final["unidade_bateu_meta"] == "Sim") &
        (final["share_nmrr_real"] >= (final["trava_dinamica_hc"] - 0.001)) &
        (final["score_R5_vs_media_cluster_simples"] >= META_VS_MEDIA_MINIMA)
    )

    final["TopTalent_Reason_R5"] = final.apply(
        lambda x: definir_motivo_generico(
            x,
            "score_R5_vs_media_cluster_ponderado",
            META_VS_MEDIA_MINIMA,
            "trava_dinamica_hc",
            "share_nmrr_real",
            "Leads",
            x["unidade_bateu_meta"]
        ),
        axis=1
    )

    final["TopTalent_Reason_R5_simple"] = final.apply(
        lambda x: definir_motivo_generico(
            x,
            "score_R5_vs_media_cluster_simples",
            META_VS_MEDIA_MINIMA,
            "trava_dinamica_hc",
            "share_nmrr_real",
            "Leads",
            x["unidade_bateu_meta"]
        ),
        axis=1
    )

    final["business_rule_share"] = final["trava_dinamica_hc"]

    return final


# =====================================================
# 3. MONTHLY CONSOLIDATION
# =====================================================

def gerar_consolidado_mensal(df_origem, perfil, pesos_dict, df_metas_unidade, df_p10_raw=None):
    """
    Generates a monthly summary:
    - Total active employees
    - Number of Top Talents per scoring model
    """

    indicadores = list(pesos_dict.keys())

    base_m = df_origem.pivot_table(
        index=[COL_COLABORADOR, COL_FRANQUIA, COL_CLUSTER, COL_PERIODO],
        columns=COL_INDICADOR,
        values=COL_RESULTADO,
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    indicadores_reais = [i for i in indicadores if i in base_m.columns]

    # Score calculations per month
    for ind in indicadores_reais:
        base_m[f"pct_{ind}"] = base_m.groupby([COL_CLUSTER, COL_PERIODO])[ind].rank(pct=True) * 100
        media_m = base_m.groupby([COL_CLUSTER, COL_PERIODO])[ind].transform('mean')
        base_m[f"vsm_{ind}"] = (base_m[ind] / (media_m.replace(0, 1))) * 100
        base_m[f"norm_{ind}"] = base_m.groupby([COL_CLUSTER, COL_PERIODO])[ind].transform(
            lambda x: (x / (x.max() if x.max() != 0 else 1) * 100)
        )

    # Final scores
    base_m["R1"] = sum(base_m[f"pct_{ind}"] * pesos_dict[ind] for ind in indicadores)
    base_m["R2"] = base_m[[f"pct_{ind}" for ind in indicadores]].mean(axis=1)
    base_m["R3"] = sum(base_m[f"norm_{ind}"] * pesos_dict[ind] for ind in indicadores)
    base_m["R4"] = base_m[[f"norm_{ind}" for ind in indicadores]].mean(axis=1)
    base_m["R5_P"] = sum(base_m[f"vsm_{ind}"] * pesos_dict[ind] for ind in indicadores)

    # Merge with targets
    base_m = base_m.merge(
        df_metas_unidade[[COL_FRANQUIA, COL_PERIODO, "bateu_meta_mes"]],
        on=[COL_FRANQUIA, COL_PERIODO],
        how="left"
    )

    hc_m = base_m.groupby([COL_FRANQUIA, COL_PERIODO])[COL_COLABORADOR].transform('nunique')

    # Share logic (SDR vs others)
    if perfil == "SDR" and df_p10_raw is not None:
        p10_c = df_p10_raw.copy()
        p10_c[COL_FRANQUIA] = p10_c[COL_FRANQUIA].astype(str).str.replace(r"^omie\s+", "", regex=True, case=False).str.strip()

        unidade_leads_m = p10_c.groupby([COL_FRANQUIA, COL_PERIODO])["Leads Agendados"] \
            .sum().reset_index(name="total_leads_unid_m")

        base_m = base_m.merge(unidade_leads_m, on=[COL_FRANQUIA, COL_PERIODO], how="left").fillna(0)

        base_m["share_m"] = np.where(
            base_m["total_leads_unid_m"] > 0,
            base_m["Leads Agendados"] / base_m["total_leads_unid_m"],
            0
        )

        base_m["trava_geral_m"] = np.where(hc_m == 1, TETO_SOLO_SDR, SHARE_FIXO_SDR)
        base_m["trava_hc_m"] = (1.0 / hc_m).clip(upper=TETO_SOLO_SDR)

    else:
        nmrr_ind = next((k for k in ["NMRR", "NMRR de Contador"] if k in base_m.columns), None)

        unid_total_m = base_m.groupby([COL_FRANQUIA, COL_PERIODO])[nmrr_ind].transform('sum') if nmrr_ind else 0

        base_m["share_m"] = np.where(unid_total_m > 0, base_m[nmrr_ind] / unid_total_m, 0)
        base_m["trava_geral_m"] = np.where(hc_m == 1, TETO_SOLO, SHARE_FIXO_PADRAO)
        base_m["trava_hc_m"] = (1.0 / hc_m).clip(upper=TETO_SOLO)

    # Month label
    base_m["Mes_Nome"] = base_m[COL_PERIODO].dt.strftime('%Y-%m')

    # Aggregation
    return base_m.groupby(['Mes_Nome']).agg(
        Total_Active=(COL_COLABORADOR, "nunique"),
        TopTalents_R1=("R1", lambda x: ((x >= SCORE_MINIMO) & (base_m.loc[x.index, "bateu_meta_mes"] == True) & (base_m.loc[x.index, "share_m"] >= (base_m.loc[x.index, "trava_geral_m"] - 0.001))).sum()),
        TopTalents_R2=("R2", lambda x: ((x >= SCORE_MINIMO) & (base_m.loc[x.index, "bateu_meta_mes"] == True) & (base_m.loc[x.index, "share_m"] >= (base_m.loc[x.index, "trava_geral_m"] - 0.001))).sum()),
        TopTalents_R3=("R3", lambda x: ((x >= SCORE_MINIMO) & (base_m.loc[x.index, "bateu_meta_mes"] == True) & (base_m.loc[x.index, "share_m"] >= (base_m.loc[x.index, "trava_geral_m"] - 0.001))).sum()),
        TopTalents_R4=("R4", lambda x: ((x >= SCORE_MINIMO) & (base_m.loc[x.index, "bateu_meta_mes"] == True) & (base_m.loc[x.index, "share_m"] >= (base_m.loc[x.index, "trava_geral_m"] - 0.001))).sum()),
        TopTalents_R5_General=("R5_P", lambda x: ((x >= META_VS_MEDIA_MINIMA) & (base_m.loc[x.index, "bateu_meta_mes"] == True) & (base_m.loc[x.index, "share_m"] >= (base_m.loc[x.index, "trava_geral_m"] - 0.001))).sum()),
        TopTalents_R5_HC=("R5_P", lambda x: ((x >= META_VS_MEDIA_MINIMA) & (base_m.loc[x.index, "bateu_meta_mes"] == True) & (base_m.loc[x.index, "share_m"] >= (base_m.loc[x.index, "trava_hc_m"] - 0.001))).sum())
    ).reset_index().assign(profile=perfil)

"""
====================================================
DATA PREPARATION, FILTERING AND FINAL PIPELINE
====================================================

This section handles:
- Data cleaning and standardization
- Time filtering (including inactive employees)
- Unit target calculations
- Full processing pipeline (scores + eligibility)
- Monthly detailed and consolidated outputs
- Audit dataset generation

Key idea:
This is the orchestration layer that connects all previous functions
into a unified and consistent scoring pipeline.
"""

# =====================================================
# 1. STANDARDIZE FRANCHISE NAMES
# =====================================================

# Removes prefix (e.g., "omie ") and trims spaces
for df in [df_evs, df_ecs, df_sdrs, df_metas]:
    if COL_FRANQUIA in df.columns:
        df[COL_FRANQUIA] = (
            df[COL_FRANQUIA]
            .astype(str)
            .str.replace(r"^omie\s+", "", regex=True, case=False)
            .str.strip()
        )

# =====================================================
# 2. DATE FILTER (INCLUDING INACTIVE EMPLOYEES)
# =====================================================

# Filter only by date range (status filter intentionally removed)
df_evs = df_evs[df_evs[COL_PERIODO].between(DATA_INICIO, DATA_FIM)].copy()
df_ecs = df_ecs[df_ecs[COL_PERIODO].between(DATA_INICIO, DATA_FIM)].copy()
df_sdrs = df_sdrs[df_sdrs[COL_PERIODO].between(DATA_INICIO, DATA_FIM)].copy()
df_p10 = df_p10[df_p10[COL_PERIODO].between(DATA_INICIO, DATA_FIM)].copy()
df_metas = df_metas[df_metas[COL_PERIODO].between(DATA_INICIO, DATA_FIM)].copy()

# =====================================================
# 3. UNIT TARGET CALCULATION
# =====================================================

# Convert to numeric (safe conversion)
df_metas[COL_NMRR_UNIDADE] = pd.to_numeric(df_metas[COL_NMRR_UNIDADE], errors="coerce")
df_metas[COL_META_COMERCIAL] = pd.to_numeric(df_metas[COL_META_COMERCIAL], errors="coerce")

# Monthly target achievement (True/False)
df_metas["bateu_meta_mes"] = (
    df_metas[COL_NMRR_UNIDADE] /
    (df_metas[COL_META_COMERCIAL].replace(0, 1))
) >= 1

# Aggregate per franchise
meta_periodo = df_metas.groupby(COL_FRANQUIA, as_index=False).agg(
    meses_total=(COL_PERIODO, "nunique"),
    meses_com_meta=("bateu_meta_mes", "sum"),
    nmrr_unidade_total=(COL_NMRR_UNIDADE, "sum")
)

# Define if unit achieved target in the period
meta_periodo["unidade_bateu_meta"] = np.where(
    (meta_periodo["meses_com_meta"] / meta_periodo["meses_total"]) >= PCT_META_MINIMO,
    "Yes",
    "No"
)

# =====================================================
# 4. UNIFIED PROCESSING PIPELINE
# =====================================================

# -----------------------------
# 4.0 CONSOLIDATED HEADCOUNT (EV + EC)
# -----------------------------
df_hc_vendas = pd.concat([
    df_evs[[COL_FRANQUIA, COL_COLABORADOR]],
    df_ecs[[COL_FRANQUIA, COL_COLABORADOR]]
]).drop_duplicates()

hc_consolidado_vendas = (
    df_hc_vendas.groupby(COL_FRANQUIA)[COL_COLABORADOR]
    .nunique()
    .reset_index(name='hc_total_unidade')
)

# -----------------------------
# 4.1 CLUSTER BENCHMARKS
# -----------------------------
medias_ev = get_medias_cluster(df_evs, list(PESOS_EVS.keys()), "EV")
medias_ec = get_medias_cluster(df_ecs, list(PESOS_ECS.keys()), "EC")
medias_sdr = get_medias_cluster(df_sdrs, list(PESOS_SDRS.keys()), "SDR")

# Save benchmark history
df_medias_cluster_atual = pd.concat([medias_ev, medias_ec, medias_sdr], ignore_index=True)
df_medias_cluster_atual.to_csv(ARQUIVO_HISTORICO_MEDIAS, index=False, encoding='utf-8-sig')

# -----------------------------
# 4.2 SCORE CALCULATION
# -----------------------------
score_evs_raw, _ = calcular_todos_racionais(df_evs, PESOS_EVS, medias_ev)
score_ecs_raw, _ = calcular_todos_racionais(df_ecs, PESOS_ECS, medias_ec)
score_sdrs_raw, _ = calcular_todos_racionais(df_sdrs, PESOS_SDRS, medias_sdr)

# -----------------------------
# 4.3 FINAL DATASETS (ELIGIBILITY + TOP TALENT)
# -----------------------------
final_evs = preparar_final_consolidado(
    score_evs_raw, df_evs, "EV", "NMRR", meta_periodo, hc_consolidado_vendas
)

final_ecs = preparar_final_consolidado(
    score_ecs_raw, df_ecs, "EC", "Accountant NMRR", meta_periodo, hc_consolidado_vendas
)

final_sdrs = preparar_final_sdrs_r7(
    score_sdrs_raw, df_sdrs, df_p10, "SDR", meta_periodo
)

# =====================================================
# 5. CONTINUITY FILTER (3-MONTH RULE)
# =====================================================

# Count active months per employee
contagem_meses_ev = df_evs.groupby(COL_COLABORADOR)[COL_PERIODO].nunique()
contagem_meses_ec = df_ecs.groupby(COL_COLABORADOR)[COL_PERIODO].nunique()
contagem_meses_sdr = df_sdrs.groupby(COL_COLABORADOR)[COL_PERIODO].nunique()

# Select employees present in all 3 months
maratonistas_ev = contagem_meses_ev[contagem_meses_ev == 3].index
maratonistas_ec = contagem_meses_ec[contagem_meses_ec == 3].index
maratonistas_sdr = contagem_meses_sdr[contagem_meses_sdr == 3].index

todos_maratonistas = list(maratonistas_ev) + list(maratonistas_ec) + list(maratonistas_sdr)

# Apply filter
resultado_trimestral_bruto = pd.concat([final_evs, final_ecs, final_sdrs], ignore_index=True)
resultado_trimestral = resultado_trimestral_bruto[
    resultado_trimestral_bruto[COL_COLABORADOR].isin(todos_maratonistas)
].copy()

# =====================================================
# 6. MONTHLY DETAILED PROCESSING
# =====================================================

def obter_detalhe_mensal_convergente():
    """
    Generates monthly detailed dataset using the SAME logic as the main pipeline.
    Ensures consistency across all calculations.
    """

    lista_detalhe = []
    meses_totais = sorted(df_metas[COL_PERIODO].unique())

    for mes in meses_totais:

        # Filter data by month
        ev_m = df_evs[df_evs[COL_PERIODO] == mes]
        ec_m = df_ecs[df_ecs[COL_PERIODO] == mes]
        sdr_m = df_sdrs[df_sdrs[COL_PERIODO] == mes]

        # Monthly headcount (EV + EC)
        hc_m_vendas = pd.concat([
            ev_m[[COL_FRANQUIA, COL_COLABORADOR]],
            ec_m[[COL_FRANQUIA, COL_COLABORADOR]]
        ]).drop_duplicates()

        hc_m_ref = hc_m_vendas.groupby(COL_FRANQUIA)[COL_COLABORADOR] \
            .nunique().reset_index(name='hc_total_unidade')

        # Monthly target summary
        meta_m_resumo = df_metas[df_metas[COL_PERIODO] == mes].groupby(COL_FRANQUIA).agg(
            nmrr_unidade_total=(COL_NMRR_UNIDADE, "sum"),
            meses_com_meta=("bateu_meta_mes", "sum")
        ).reset_index()

        meta_m_resumo["unidade_bateu_meta"] = np.where(
            meta_m_resumo["meses_com_meta"] >= 1,
            "Yes",
            "No"
        )

        # Process EV and EC
        for df_m, perf, pesos, col_ind in [
            (ev_m, "EV", PESOS_EVS, "NMRR"),
            (ec_m, "EC", PESOS_ECS, "Accountant NMRR")
        ]:
            if not df_m.empty:
                bench_m = get_medias_cluster(df_m, list(pesos.keys()), perf)
                score_m_raw, _ = calcular_todos_racionais(df_m, pesos, bench_m)
                f_m = preparar_final_consolidado(score_m_raw, df_m, perf, col_ind, meta_m_resumo, hc_m_ref)
                f_m["Mes_Name"] = pd.to_datetime(mes).strftime('%Y-%m')
                lista_detalhe.append(f_m)

        # Process SDR
        if not sdr_m.empty:
            p10_m = df_p10[df_p10[COL_PERIODO] == mes]
            bench_sdr_m = get_medias_cluster(sdr_m, list(PESOS_SDRS.keys()), "SDR")
            score_sdr_m_raw, _ = calcular_todos_racionais(sdr_m, PESOS_SDRS, bench_sdr_m)
            f_sdr_m = preparar_final_sdrs_r7(score_sdr_m_raw, sdr_m, p10_m, "SDR", meta_m_resumo)
            f_sdr_m["Mes_Name"] = pd.to_datetime(mes).strftime('%Y-%m')
            lista_detalhe.append(f_sdr_m)

    return pd.concat(lista_detalhe, ignore_index=True)


# Generate final datasets
df_detalhe_mensal = obter_detalhe_mensal_convergente()

# =====================================================
# 7. CONSOLIDATED MONTHLY TABLE
# =====================================================

df_consolidado_mensal = df_detalhe_mensal.groupby(['Mes_Nome', 'perfil']).agg(
    Total_Active=(COL_COLABORADOR, "nunique"),
    TopTalents_R1=("Elegivel_R1", lambda x: (x == "Eligible").sum()),
    TopTalents_R2=("Elegivel_R2", lambda x: (x == "Eligible").sum()),
    TopTalents_R3=("Elegivel_R3", lambda x: (x == "Eligible").sum()),
    TopTalents_R4=("Elegivel_R4", lambda x: (x == "Eligible").sum()),
    TopTalents_R5_HC=("TopTalent_Official", "sum"),
    TopTalents_R5_General=("TopTalent_Official", "sum")
).reset_index().sort_values(['Mes_Nome', 'perfil'])


# =====================================================
# 8. AUDIT TABLE GENERATION
# =====================================================

def gerar_aba_auditoria(final_df, score_raw, pesos_dict, perfil_nome):
    """
    Creates a technical audit dataset showing how scores were built.

    Includes:
    - Raw KPI values
    - Percentiles
    - Normalized scores
    - Benchmark comparisons
    """

    indicadores = list(pesos_dict.keys())

    cols_base = [COL_COLABORADOR, COL_FRANQUIA, COL_CLUSTER, COL_STATUS]
    cols_auditoria = cols_base.copy()

    for ind in indicadores:
        for c in [
            f"{ind}_colab",
            f"pct_{ind}",
            f"norm_{ind}",
            f"{ind}_vs_Media_Cluster"
        ]:
            if c in score_raw.columns:
                cols_auditoria.append(c)

    df_auditoria = score_raw[cols_auditoria].copy()
    df_auditoria["Profile"] = perfil_nome

    return df_auditoria

"""
====================================================
AUDIT DATA, EXPORT AND FINAL REPORT GENERATION
====================================================

This section handles:
- Enriching the audit dataset with business rules (share + thresholds)
- Consolidating audit data across all roles
- Preparing final reporting views (R5 logic variations)
- Calculating retention metrics
- Exporting a structured Excel report with multiple tabs

Key idea:
This is the final delivery layer, transforming all processed data
into business-ready outputs and audit transparency.
"""

# =====================================================
# 1. ENRICH AUDIT DATA WITH BUSINESS RULES
# =====================================================

# Add share and threshold information to audit dataset
df_travas = final_df[
    [COL_COLABORADOR, "share_nmrr_real", "trava_fixa", "trava_dinamica_hc", "unidade_bateu_meta"]
]

df_auditoria = df_auditoria.merge(df_travas, on=COL_COLABORADOR, how="left")

return df_auditoria


# =====================================================
# 2. GENERATE AUDIT DATASETS
# =====================================================

auditoria_ev = gerar_aba_auditoria(final_evs, score_evs_raw, PESOS_EVS, "EV")
auditoria_ec = gerar_aba_auditoria(final_ecs, score_ecs_raw, PESOS_ECS, "EC")
auditoria_sdr = gerar_aba_auditoria(final_sdrs, score_sdrs_raw, PESOS_SDRS, "SDR")

# Consolidated audit dataset
df_auditoria_consolidada = pd.concat(
    [auditoria_ev, auditoria_ec, auditoria_sdr],
    ignore_index=True
)

# Flag: completed full quarter (3 months)
df_auditoria_consolidada["Completed_Quarter"] = np.where(
    df_auditoria_consolidada[COL_COLABORADOR].isin(todos_maratonistas),
    "Yes",
    "No"
)

# Debug: distribution of months per employee
contagem = df_evs.groupby(COL_COLABORADOR)[COL_PERIODO].nunique()
print(contagem.value_counts())


# =====================================================
# 3. EXPORT PREPARATION (R5 STRUCTURE)
# =====================================================

# Standard columns for R5 reporting
cols_r5_exp = [
    COL_COLABORADOR, COL_STATUS, COL_FRANQUIA, COL_CLUSTER, "perfil",
    "unidade_bateu_meta", "share_nmrr_real", "share_regra_negocio",
    "score_R5_vs_media_cluster_ponderado", "TopTalent_Official",
    "TopTalent_Reason_R5", "score_R5_vs_media_cluster_simples",
    "TopTalent_Official_simple", "TopTalent_Reason_R5_simple"
]

cols_detalhe_mes = ["Mes_Nome"] + cols_r5_exp


# =====================================================
# 4. BUSINESS RULE THRESHOLD FUNCTION
# =====================================================

def calcular_trava_geral(row):
    """
    Defines fixed threshold based on role and headcount.
    """
    hc_val = row.get("hc_total_unidade") if pd.notna(row.get("hc_total_unidade")) else 1

    if row["perfil"] == "SDR":
        return TETO_SOLO_SDR if hc_val <= 1 else SHARE_FIXO_SDR
    else:
        return TETO_SOLO if hc_val <= 1 else SHARE_FIXO_PADRAO


# =====================================================
# 5. R5 GENERAL VIEW (FIXED BUSINESS RULE)
# =====================================================

df_geral_r5 = resultado_trimestral.copy()

# Apply fixed rule threshold
df_geral_r5["business_rule_share_general"] = df_geral_r5.apply(calcular_trava_geral, axis=1)

# Recalculate eligibility using fixed threshold
df_geral_r5["TopTalent_Official"] = (
    (df_geral_r5["unidade_bateu_meta"] == "Sim") &
    (df_geral_r5["share_nmrr_real"] >= (df_geral_r5["business_rule_share_general"] - 0.001)) &
    (df_geral_r5["score_R5_vs_media_cluster_ponderado"] >= META_VS_MEDIA_MINIMA)
)

df_geral_r5["TopTalent_Official_simple"] = (
    (df_geral_r5["unidade_bateu_meta"] == "Sim") &
    (df_geral_r5["share_nmrr_real"] >= (df_geral_r5["business_rule_share_general"] - 0.001)) &
    (df_geral_r5["score_R5_vs_media_cluster_simples"] >= META_VS_MEDIA_MINIMA)
)

# Update fields
df_geral_r5["share_regra_negocio"] = df_geral_r5["business_rule_share_general"]

# Reasons
df_geral_r5["TopTalent_Reason_R5"] = df_geral_r5.apply(
    lambda x: definir_motivo_generico(
        x,
        "score_R5_vs_media_cluster_ponderado",
        META_VS_MEDIA_MINIMA,
        "share_regra_negocio",
        "share_nmrr_real",
        "Indicator",
        x["unidade_bateu_meta"]
    ),
    axis=1
)

df_geral_r5["TopTalent_Reason_R5_simple"] = df_geral_r5.apply(
    lambda x: definir_motivo_generico(
        x,
        "score_R5_vs_media_cluster_simples",
        META_VS_MEDIA_MINIMA,
        "share_regra_negocio",
        "share_nmrr_real",
        "Indicator",
        x["unidade_bateu_meta"]
    ),
    axis=1
)


# =====================================================
# 6. R5 HC VIEW (DYNAMIC 1/N THRESHOLD)
# =====================================================

# Already calculated using dynamic threshold in previous steps
df_hc = resultado_trimestral.copy()


# =====================================================
# 7. RETENTION CALCULATION
# =====================================================

df_tt_hc = df_hc[df_hc["TopTalent_Official"] == True].copy()

if not df_tt_hc.empty:
    df_tt_hc["Is_Active"] = df_tt_hc[COL_STATUS].str.contains("ATIVO", na=False, case=False)

    df_retencao = df_tt_hc.groupby("perfil").agg(
        Total_Top_Talents=(COL_COLABORADOR, "count"),
        Active=("Is_Active", "sum"),
        Inactive=("Is_Active", lambda x: len(x) - x.sum())
    ).reset_index()

    df_retencao["Retention_%"] = (
        df_retencao["Active"] /
        df_retencao["Total_Top_Talents"].replace(0, 1)
    ) * 100

else:
    df_retencao = pd.DataFrame(
        columns=["perfil", "Total_Top_Talents", "Active", "Inactive", "Retention_%"]
    )


# =====================================================
# 8. FINAL EXPORT TO EXCEL
# =====================================================

with pd.ExcelWriter("Performance_Report_Q1_2026.xlsx", engine="xlsxwriter") as writer:

    workbook = writer.book

    # Formatting
    fmt_header = workbook.add_format({'bold': True, 'bg_color': '#D7E4BC', 'border': 1})
    fmt_text = workbook.add_format({'text_wrap': True, 'border': 1})

    # Monthly views
    df_consolidado_mensal.to_excel(writer, sheet_name="Monthly_TopTalent_Summary", index=False)
    df_detalhe_mensal[cols_detalhe_mes].to_excel(writer, sheet_name="Monthly_TopTalent_Detail", index=False)

    # Audit tab
    df_auditoria_consolidada.to_excel(writer, sheet_name="Audit_Detailed_Calculation", index=False)

    # General views
    resultado_trimestral.to_excel(writer, sheet_name="General_Summary", index=False)
    df_geral_r5[cols_r5_exp].to_excel(writer, sheet_name="R5_General_View", index=False)
    df_hc[cols_r5_exp].to_excel(writer, sheet_name="R5_HC_View", index=False)

    # Retention
    df_retencao.to_excel(writer, sheet_name="Top_Performer_Retention", index=False)

    # R1–R4 eligibility
    cols_r1_r4 = [
        COL_COLABORADOR, COL_STATUS, COL_FRANQUIA, COL_CLUSTER, "nmrr_total_colab",
        "score_R1_percentil_ponderado", "Elegivel_R1", "Motivo_Status_Elegibilidade_R1",
        "score_R2_percentil_simples", "Elegivel_R2",
        "score_R3_direto_ponderado", "Elegivel_R3",
        "score_R4_direto_simples", "Elegivel_R4"
    ]

    final_evs[final_evs.columns.intersection(cols_r1_r4)].to_excel(writer, sheet_name="Eligibility_R1_R4_EV", index=False)
    final_ecs[final_ecs.columns.intersection(cols_r1_r4)].to_excel(writer, sheet_name="Eligibility_R1_R4_EC", index=False)
    final_sdrs[final_sdrs.columns.intersection(cols_r1_r4)].to_excel(writer, sheet_name="Eligibility_R1_R4_SDR", index=False)

    # Benchmarks
    pd.concat([medias_ev, medias_ec, medias_sdr]).to_excel(writer, sheet_name="Cluster_Benchmarks", index=False)

    # Methodology
    ws_meto = writer.book.add_worksheet("Methodology")

    metodologia_dados = [
        ['1. Unit Eligibility', 'Franchise must achieve target in at least 60% of months.'],
        ['2. R5 General View', 'Applies fixed share threshold (role-based).'],
        ['3. R5 HC View', 'Applies dynamic threshold based on headcount (1/N).'],
        ['4. R1–R4 Models', 'Ranking (percentile) and normalized scoring.'],
        ['5. R5 Model', 'Compares individual vs cluster average (>=100).'],
        ['6. Retention', 'Measures how many Top Talents remain active.'],
        ['7. Monthly View', 'Detailed monthly performance breakdown.']
    ]

    for i, (topic, rule) in enumerate(metodologia_dados):
        ws_meto.write(i + 2, 0, topic, fmt_header)
        ws_meto.write(i + 2, 1, rule, fmt_text)

print("Report successfully generated!")